# 19 · Exportar EmoGol-2B como archivo ejecutable (`emogol.joblib`)

Este notebook convierte el modelo de los notebooks en un **único archivo** que cualquier aplicación puede cargar con `joblib.load`.

El archivo contiene todo lo propio de EmoGol-2B:

- la configuración de uso: plantilla del prompt, recorte a 112 tokens, decodificación voraz con 6 tokens y lotes de 16;
- las reglas de entrada (filtro de validez y antispam) y la regla de lectura de la respuesta;
- el adaptador LoRA completo (`gemma2_2b/adapter_final`, unos 112 MiB).

El modelo base Gemma-2 2B de Google no va dentro: se descarga de Hugging Face al cargar el archivo. Así el `.joblib` pesa unos 112 MiB en lugar de 2 GB y no se rompe al cambiar la versión de PyTorch o bitsandbytes.

**Entorno:** Google Colab con GPU T4. **Secretos:** `HF_TOKEN` (con permiso de escritura).

**Resultado:** `emogol.joblib`, `emogol.json` y la tarjeta del modelo, publicados en el repositorio de Hugging Face `santiagortegasegura/emogol-2b`, junto con una copia del código de la aplicación.

In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes "huggingface_hub<2.0" "pandas<3" joblib
!pip uninstall -y -q torchao
!rm -rf emotion_futbol && git clone -q https://github.com/santiagortegasegura/emotion_futbol.git
!pip install -q -e emotion_futbol/emogol_app

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for emogol (pyproject.toml) ... done


In [2]:
import os, sys
from google.colab import userdata
from huggingface_hub import login

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])

USUARIO_HF = "santiagortegasegura"
REPO_MODELO = f"{USUARIO_HF}/emogol-2b"           # repositorio nuevo, tipo modelo
REPO_DATOS = f"{USUARIO_HF}/emotion-futbol-data"
sys.path.insert(0, "emotion_futbol/emogol_app")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 1. Construir el archivo

`exportar_joblib.py` baja el adaptador de `emotion-futbol-checkpoints`, lo guarda junto con la configuración dentro de un objeto `EmoGolClasificador` y escribe `emogol.joblib`.

In [3]:
from exportar_joblib import exportar
ruta = exportar("emogol.joblib")
!ls -lh emogol.joblib emogol.json

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Adaptador: 6 archivos, 112.0 MiB
Guardado: emogol.joblib (44.4 MiB) | huella 901218559d5ff61d
-rw-r--r-- 1 root root  45M Nov 17 16:44 emogol.joblib
-rw-r--r-- 1 root root 1.4K Nov 17 16:44 emogol.json


## 2. Comprobar que el archivo funciona

Se carga el archivo como lo haría la aplicación, sin usar el código de los notebooks anteriores.

In [4]:
import time, joblib, torch

t0 = time.time()
emogol = joblib.load("emogol.joblib")
emogol.cargar()
print(f"Cargado en {time.time() - t0:.0f} s | GPU {torch.cuda.get_device_name(0)} | "
      f"memoria {torch.cuda.memory_allocated() / 2**30:.2f} GiB")
print(emogol.info())

ejemplos = ["LO BOTO JAJAJAJA", "qué robo del árbitro, siempre lo mismo", "Felicitaciones España, merecido campeón",
            "tapalo tapalo por favor", "ufff qué nervios, no puedo ver los penales"]
for texto, emocion in zip(ejemplos, emogol.predecir_nombres(ejemplos)):
    print(f"{emocion:<10} {texto}")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


Cargado en 82 s | GPU Tesla T4 | memoria 2.15 GiB
{'version': '1.0.0', 'modelo_base': 'google/gemma-2-2b-it', 'adaptador': 'santiagortegasegura/emotion-futbol-checkpoints/gemma2_2b/adapter_final', 'tamano_adaptador_mib': 112.0, 'huella_adaptador': '901218559d5ff61d', 'cargado': True, 'creado': '2026-11-17T16:44:22+00:00', 'origen': 'https://huggingface.co/datasets/santiagortegasegura/emotion-futbol-checkpoints'}
Desprecio  LO BOTO JAJAJAJA
Enojo      qué robo del árbitro, siempre lo mismo
Alegría    Felicitaciones España, merecido campeón
Enojo      tapalo tapalo por favor
Miedo      ufff qué nervios, no puedo ver los penales


## 3. Verificar que reproduce la evaluación

Se clasifican 1.000 comentarios de la prueba final, elegidos al azar con la semilla 42. La accuracy debe quedar cerca de 0,825; con 1.000 comentarios el margen es de unos ±2,4 puntos. Si queda muy por fuera, el archivo no reproduce el modelo evaluado y no debe publicarse.

In [5]:
import pandas as pd
from huggingface_hub import hf_hub_download

prueba = pd.read_csv(hf_hub_download(REPO_DATOS, "final/prueba_final_10k.csv", repo_type="dataset"),
                     keep_default_na=False)
muestra = prueba.sample(1000, random_state=42)

torch.cuda.synchronize(); t0 = time.time()
muestra["prediccion"] = emogol.predecir(muestra["text"].tolist())
torch.cuda.synchronize(); segundos = time.time() - t0

accuracy = (muestra["prediccion"] == muestra["label"]).mean()
invalidas = (muestra["prediccion"] == "invalido").mean()
print(f"Accuracy: {accuracy:.3f} (evaluación completa: 0,825) | inválidas: {invalidas:.1%}")
print(f"Velocidad: {len(muestra) / segundos:.2f} comentarios/s | memoria pico {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
assert 0.79 <= accuracy <= 0.86, "La accuracy no corresponde a EmoGol-2B: revisar antes de publicar."
print("Verificación: OK")

prueba_final_10k.csv:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Accuracy: 0.852 (evaluación completa: 0,825) | inválidas: 0.0%
Velocidad: 3.62 comentarios/s | memoria pico 2.57 GiB
Verificación: OK


## 4. Publicar en Hugging Face

Se crea el repositorio de modelo `emogol-2b` (público) con:

- `emogol.joblib` y `emogol.json` (descripción del archivo y huella del adaptador);
- `README.md`, la tarjeta del modelo;
- `codigo/`, copia completa del repositorio de GitHub en esta versión (notebooks, aplicación y documentación). La carpeta `codigo/emogol_app` es la que se necesita para cargar el archivo.

In [6]:
from huggingface_hub import create_repo, upload_file, upload_folder
import json

info = json.load(open("emogol.json"))
tarjeta = f"""---
language: es
license: gemma
base_model: google/gemma-2-2b-it
library_name: peft
tags: [emotion-classification, spanish, football, qlora, joblib]
---
# EmoGol-2B (archivo ejecutable)

EmoGol-2B clasifica comentarios de fútbol en español en siete emociones: alegría, tristeza, enojo,
miedo, sorpresa, desprecio y neutral. Es Gemma-2 2B ajustado con QLoRA sobre 100.000 comentarios de YouTube.
En los datos y en el prompt, el desprecio conserva la etiqueta `burla`.

Este repositorio contiene `emogol.joblib`, un archivo con la configuración, las reglas y el adaptador LoRA
del modelo ({info["tamano_adaptador_mib"]} MiB, huella `{info["huella_adaptador"]}`). El modelo base se descarga
de `google/gemma-2-2b-it` al cargarlo, por lo que se requiere aceptar su licencia y un token de Hugging Face.

## Uso

```python
# pip install "git+https://huggingface.co/{REPO_MODELO}#subdirectory=codigo/emogol_app" (o desde el repositorio de GitHub)
import joblib
from huggingface_hub import hf_hub_download

emogol = joblib.load(hf_hub_download("{REPO_MODELO}", "emogol.joblib"))
emogol.predecir_nombres(["LO BOTO JAJAJAJA", "qué robo del árbitro"])
```

Requiere una GPU NVIDIA (probado en una Tesla T4). Sobre la prueba de 10.000 comentarios, medido frente a las
etiquetas de referencia: accuracy 0,825 y F1 macro 0,776.

- Código: https://github.com/santiagortegasegura/emotion_futbol (DOI 10.5281/zenodo.23547984)
- Datos: https://huggingface.co/datasets/{REPO_DATOS} (DOI 10.57967/hf/11269)
"""
open("README_modelo.md", "w", encoding="utf-8").write(tarjeta)

create_repo(REPO_MODELO, repo_type="model", private=False, exist_ok=True)
for local, remoto in [("emogol.joblib", "emogol.joblib"), ("emogol.json", "emogol.json"), ("README_modelo.md", "README.md")]:
    upload_file(path_or_fileobj=local, path_in_repo=remoto, repo_id=REPO_MODELO, repo_type="model")
    print(f"Subido: {remoto}")
upload_folder(folder_path="emotion_futbol", path_in_repo="codigo", repo_id=REPO_MODELO, repo_type="model",
              ignore_patterns=[".git/*", "*.joblib", ".env", "**/__pycache__/*", "**/*.egg-info/*"])
print(f"Listo: https://huggingface.co/{REPO_MODELO}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/emogol.joblib      :   1%|1         |  524kB / 46.6MB            

Validating                    :           |   0%            

Subido: emogol.joblib
Subido: emogol.json
Subido: README.md


/usr/local/lib/python3.13/dist-packages/huggingface_hub/hf_api.py:11779: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Listo: https://huggingface.co/santiagortegasegura/emogol-2b
